# 02 — Engenharia de Variáveis

Etapa 2 do procedimento metodológico (Seção 3.4), primeira metade: transformar os dados brutos e heterogêneos em
variáveis candidatas para a Rede Bayesiana. As decisões vêm da EDA (notebook 00) e do diagnóstico (notebook 01).

**Princípios que guiam todas as variáveis:**

1. **Definir o instante da predição.** O motivo da falta é excluído, e o histórico usa desfechos de dias anteriores à consulta.
   Clima observado e cadastro sem versionamento exigem as ressalvas de disponibilidade do notebook 01; não se pressupõe que tudo já era conhecido na marcação.
2. **O histórico do paciente usa apenas dias anteriores.** Quando o paciente tem várias consultas no mesmo dia, em
   93% dos casos ou vai a todas ou falta a todas. Contar faltas do mesmo dia seria "colar" a resposta.
3. **Ausência é informação.** Quem não tem cadastro falta mais, então "ausente" vira uma categoria explícita, e não
   uma linha descartada.
4. **Na avaliação, os parâmetros são aprendidos no treino.** Cortes de discretização, escalonadores e SMOTENC são ajustados
   só no fold de treino (notebook 04). Aqui também há agrupamentos e associações exploratórias. A discretização é demonstrada
   na base inteira **apenas para fins de inspeção**.

O código está em `src/noshow/features.py` e é coberto por testes automatizados (`tests/test_features.py`).

**Separação entre exploração e avaliação.** As frequências e faixas calculadas neste notebook são exploratórias. A coluna original `city` é salva como auxiliar, sem ser preditor adicional, para que 03–04 reaprendam o agrupamento de cidades em cada treino. Categorias, cortes e suavização usados para medir desempenho não podem ser escolhidos pela base inteira.


**Independência das divisões.** A avaliação dos notebooks 03–04 mantém juntos os pacientes identificados e os registros indistinguíveis pelos campos originais (incluindo data/hora, excluindo alvo e motivo). Registros sem ID não são mais automaticamente independentes entre si. As linhas são preservadas; os grupos de validação não são atributos preditivos. A auditoria verifica zero sobreposição externa e interna em `resultados/artigo_sem_sobreposicao/auditoria_divisoes.csv`. A falta de ID ainda impede reconhecer a mesma pessoa em consultas diferentes.


**Discretização e relações derivadas no protocolo ajustado.** As tabelas de cinco faixas deste notebook são exploração. No 04, a escolha entre três e cinco faixas ocorre em validação interna, igualmente para controle e engenharia. A taxa de faltas anteriores é derivada das contagens: no SMOTENC ajustado ela sai do cálculo de vizinhança e é recalculada apenas nos sintéticos. Contagens sintéticas são valores interpolados, não eventos observados; sem contagens conhecidas ou com denominador zero, a taxa fica ausente. Os registros reais são preservados.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

from noshow.data import load_raw, add_patient_proxy, TARGET
from noshow.weather import fix_weather
from noshow.features import (FEATURES, CATEGORICAL, CONTINUOUS, Discretizer, add_age, add_calendar,
                             group_categories, add_service_time, add_patient_history, build_features)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)

BLUE, ORANGE, GRAY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f1ed", BLUE])
PCT = mtick.PercentFormatter(1, decimals=0)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

def cramers_v(x, y):
    tabela = pd.crosstab(x, y)
    if min(tabela.shape) < 2:
        return np.nan
    chi2 = stats.chi2_contingency(tabela)[0]
    return np.sqrt(chi2 / (tabela.to_numpy().sum() * (min(tabela.shape) - 1)))

## 1. Ponto de partida

Carga com as correções já validadas: tipos e datas (`load_raw`), clima sem artefatos e com a temperatura dos 32 dias
ausentes preenchida pelo Open-Meteo (`fix_weather`) e ID aproximado do paciente (`add_patient_proxy`).

In [ ]:
base = add_patient_proxy(fix_weather(load_raw()))
print(f"{len(base):,} agendamentos | {base.patient_id.nunique():,} pacientes identificados "
      f"({base.patient_id.notna().mean():.1%} dos registros) | no-show {base[TARGET].mean():.1%}")

## 2. Variáveis demográficas

- **Idade na data da consulta:** substitui `age`, que é a idade na extração da base. Idades ≥ 100 anos são tratadas
  como data de nascimento inválida.
- **Faixa etária:** 0–5, 6–11, 12–17, 18–59 e 60+, com "ausente" como categoria.
- **Agrupamentos:**
  - cidades com menos de 300 agendamentos → "outras";
  - CID → capítulo (F, R, I, G, Q, outros);
  - `gender = "I"` → ausente.
- **`cadastro_ausente`:** indica que todo o bloco de cadastro está vazio.

In [ ]:
df = add_age(base)
df = group_categories(df)

erro_age = (base.age - df.idade_consulta).groupby(df.appointment_date.dt.year).mean()
print("Erro médio da coluna original `age` em relação à idade na consulta, por ano (anos):")
print(erro_age.round(1).to_string())
print(f"\nIdades ≥ 100 anos descartadas: {((base.appointment_date - base.date_of_birth).dt.days / 365.25 >= 100).sum()}")

In [ ]:
def taxa_por(col, data):
    return data.groupby(col, observed=True)[TARGET].agg(taxa="mean", n="size").sort_values("taxa")

def painel(colunas, data, titulo, ncols=3):
    nlin = int(np.ceil(len(colunas) / ncols))
    fig, axes = plt.subplots(nlin, ncols, figsize=(4.4 * ncols, 3.2 * nlin))
    media = data[TARGET].mean()
    for ax, col in zip(np.ravel(axes), colunas):
        g = taxa_por(col, data)
        ax.barh(g.index.astype(str), g.taxa, color=BLUE, height=0.6)
        ax.axvline(media, color=GRAY, ls="--", lw=1)
        for i, (t, n) in enumerate(zip(g.taxa, g.n)):
            ax.text(t, i, f" {t:.1%} (n={n:,})", va="center", fontsize=7, color=INK)
        ax.set_title(col); ax.xaxis.set_major_formatter(PCT); ax.set_xlim(0, g.taxa.max() * 1.7)
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
    for ax in np.ravel(axes)[len(colunas):]:
        ax.set_visible(False)
    fig.suptitle(f"{titulo} (linha tracejada = média {media:.1%})", x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(); plt.show()

painel(["faixa_etaria", "city_grp", "cid_capitulo", "cadastro_ausente", "disability", "specialty_grp"], df,
       "Taxa de no-show — variáveis demográficas e de atendimento")

## 3. Variáveis temporais e de agenda

- **Dia da semana:** sábado e domingo ficam juntos em "fim de semana", porque são raros (1.301 registros).
- **Faixa horária:** 17h e 18h ficam **separadas**, porque são o horário com mais faltas (12,3%) e o com menos (6,8%).
  Juntá-las apagaria o sinal.
- **Mês e estação do ano:** são duas granularidades candidatas. O Manto de Markov decide qual sobrevive.
- **Período** (pré-pandemia, pandemia e 2022): representa a deriva temporal sem usar o ano diretamente. Um modelo em
  produção nunca veria "2023" no treino.

In [ ]:
df = add_calendar(df)
painel(["dia_semana", "faixa_horaria", "periodo", "estacao", "mes"], df, "Taxa de no-show — calendário")

## 4. Variáveis de vínculo e histórico do paciente

Esta é a contribuição principal deste notebook. A EDA mostrou que **10% dos pacientes concentram 42% das faltas**,
mas nenhuma variável da base original captura esse comportamento. Com o ID aproximado, é possível reconstruí-lo.

| Variável | Definição |
|---|---|
| `dias_em_servico` | dias desde a entrada no serviço (datas de entrada posteriores à consulta → ausente) |
| `consultas_no_dia` | quantas consultas o paciente tem naquele dia (conhecido no agendamento) |
| `n_agendamentos_anteriores` | consultas em **dias anteriores** |
| `n_faltas_anteriores` | faltas em **dias anteriores** |
| `taxa_faltas_anteriores` | `n_faltas_anteriores / n_agendamentos_anteriores` (ausente se não há passado) |
| `dias_desde_ultima_consulta` | intervalo até o dia de consulta anterior |
| `faltou_ultima` | faltou no dia de consulta anterior? (`sim` / `nao` / `sem_historico` / `ausente` sem ID) |

**Exemplo com um paciente real.** O histórico de cada linha só "enxerga" os dias anteriores. As duas consultas do
mesmo dia recebem o mesmo histórico, e a falta daquele dia só é contada a partir do dia seguinte.

In [ ]:
df = add_service_time(df)
df = add_patient_history(df)

cols_hist = ["appointment_date", "specialty", TARGET, "consultas_no_dia", "n_agendamentos_anteriores",
             "n_faltas_anteriores", "taxa_faltas_anteriores", "dias_desde_ultima_consulta", "faltou_ultima"]
def bom_exemplo(g):
    inicio = g.sort_values("appointment_date").head(12)
    return (inicio[TARGET].sum() >= 2 and inicio.specialty.notna().all()
            and (inicio.groupby("appointment_date").size() > 1).any())

exemplo = df.dropna(subset=["patient_id"]).groupby("patient_id").filter(bom_exemplo).patient_id.iloc[0]
df.loc[df.patient_id == exemplo, cols_hist].sort_values(["appointment_date", "specialty"]).head(12)

In [ ]:
# Verificações de sanidade sobre a base real (complementam os testes unitários)
p = df.dropna(subset=["patient_id"])
primeira = p.groupby("patient_id").appointment_date.transform("min") == p.appointment_date
assert (p.loc[primeira, "n_agendamentos_anteriores"] == 0).all(), "primeiro dia deve ter histórico vazio"
assert (p.n_faltas_anteriores <= p.n_agendamentos_anteriores).all()
mesmo_dia = p.groupby(["patient_id", "appointment_date"])[["n_faltas_anteriores", "faltou_ultima"]].nunique()
assert (mesmo_dia <= 1).all().all(), "consultas do mesmo dia devem ter o mesmo histórico"
assert len(df) == len(base), "nenhuma linha pode ser perdida ou duplicada"
print("✓ primeiro dia sem histórico | faltas ≤ agendamentos | mesmo dia = mesmo histórico | linhas preservadas")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

g = taxa_por("faltou_ultima", df).reindex(["nao", "sim", "sem_historico", "ausente"])
axes[0].bar(g.index, g.taxa, color=[BLUE, ORANGE, GRAY, GRAY], width=0.6)
for i, (t, n) in enumerate(zip(g.taxa, g.n)):
    axes[0].text(i, t, f"{t:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=7, color=INK)
axes[0].set_title("Faltou na consulta anterior?"); axes[0].set_ylim(0, g.taxa.max() * 1.35)

com_hist = df[df.n_agendamentos_anteriores >= 5]
faixas = pd.cut(com_hist.taxa_faltas_anteriores, [-0.001, 0, 0.05, 0.10, 0.20, 1],
                labels=["0%", "0–5%", "5–10%", "10–20%", ">20%"])
g = com_hist.groupby(faixas, observed=True)[TARGET].agg(taxa="mean", n="size")
axes[1].bar(g.index.astype(str), g.taxa, color=BLUE, width=0.6)
for i, (t, n) in enumerate(zip(g.taxa, g.n)):
    axes[1].text(i, t, f"{t:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=7, color=INK)
axes[1].set_title("Taxa de faltas anteriores (≥ 5 consultas)"); axes[1].set_ylim(0, g.taxa.max() * 1.35)

intervalo = pd.cut(df.dias_desde_ultima_consulta, [0, 6, 7, 14, 30, np.inf], labels=["< 7", "7", "8–14", "15–30", "> 30"])
g = df.groupby(intervalo, observed=True)[TARGET].agg(taxa="mean", n="size")
axes[2].bar(g.index.astype(str), g.taxa, color=BLUE, width=0.6)
for i, (t, n) in enumerate(zip(g.taxa, g.n)):
    axes[2].text(i, t, f"{t:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=7, color=INK)
axes[2].set_title("Dias desde a consulta anterior"); axes[2].set_ylim(0, g.taxa.max() * 1.35)

for ax in axes:
    ax.yaxis.set_major_formatter(PCT); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Controle pela deriva temporal.** A taxa de no-show cresce ao longo dos anos, e o histórico acumulado também.
O efeito do histórico poderia ser só o tempo disfarçado. A tabela abaixo compara `faltou_ultima` **dentro de cada
período**.

In [ ]:
controle = (df[df.faltou_ultima.isin(["nao", "sim"])]
            .pivot_table(index="periodo", columns="faltou_ultima", values=TARGET, aggfunc="mean"))
controle["razão sim/não"] = controle["sim"] / controle["nao"]
controle.style.format({"nao": "{:.1%}", "sim": "{:.1%}", "razão sim/não": "{:.2f}×"})

## 5. Catálogo de variáveis candidatas

Todas as variáveis candidatas, com o tipo que orienta o pipeline: **categóricas** entram direto na Rede Bayesiana e
no SMOTENC como nominais; **contínuas** passam pelo escalonamento (para o SMOTENC) e pela discretização (para a RB).

In [ ]:
catalogo = pd.DataFrame([
    {"variável": nome, "tipo": tipo, "grupo (Seção 2.1.4)": grupo,
     "categorias / faixa": (f"{df[nome].nunique()} categorias" if tipo == "categorical"
                            else f"{df[nome].min():g} a {df[nome].max():g}"),
     "ausente_%": 100 * (df[nome].isna() | (df[nome].astype("string") == "ausente")).mean()}
    for nome, (tipo, grupo) in FEATURES.items()
]).set_index("variável")
print(f"{len(CATEGORICAL)} categóricas + {len(CONTINUOUS)} contínuas = {len(FEATURES)} variáveis candidatas")
catalogo.style.format({"ausente_%": "{:.1f}"})

## 6. Discretização para a Rede Bayesiana

Redes Bayesianas discretas trabalham com estados, não com números. Cada variável contínua é cortada em até 5 faixas
de mesma frequência (quantis), e os valores ausentes ganham a faixa própria "ausente". Variáveis com muitos zeros
(como `n_faltas_anteriores`) acabam com menos faixas, porque cortes repetidos são unificados.

> No experimento (notebook 04), o `Discretizer` é ajustado **dentro de cada fold de treino**. Aqui ele é ajustado na
> base inteira só para inspecionar as faixas resultantes.

In [ ]:
disc = Discretizer(columns=CONTINUOUS, n_bins=5).fit(df)
discretizada = disc.transform(df[CATEGORICAL + CONTINUOUS])

faixas = pd.DataFrame({
    col: {"cortes": ", ".join(f"{e:.3g}" for e in disc.edges_[col]),
          "nº de estados": discretizada[col].nunique(),
          "taxa por estado": " | ".join(f"{k}: {v:.1%}" for k, v in
                                        df.groupby(discretizada[col])[TARGET].mean().reindex(disc.labels_[col]).dropna().items())}
    for col in CONTINUOUS
}).T
faixas

In [ ]:
estados = pd.Series({c: discretizada[c].nunique() for c in discretizada})
print(f"Estados por variável: mín {estados.min()}, mediana {estados.median():.0f}, máx {estados.max()} ({estados.idxmax()})")
print(f"Uma CPT de no-show com 3 pais de 6 estados já teria {6**3:,} combinações → motivo para o Manto de Markov "
      "limitar as variáveis; Noisy-OR não é aplicado neste código")

## 7. Poder preditivo e redundância

**Associação com o no-show:** V de Cramér de cada variável **já discretizada** com o alvo, comparando o ranking com
o das variáveis originais.

In [ ]:
assoc = pd.DataFrame({
    "V": {c: cramers_v(discretizada[c], df[TARGET]) for c in discretizada},
    "grupo": {c: FEATURES[c][1] for c in discretizada},
    "nova": {c: c in ["n_agendamentos_anteriores", "n_faltas_anteriores", "taxa_faltas_anteriores",
                      "dias_desde_ultima_consulta", "faltou_ultima", "consultas_no_dia", "cadastro_ausente",
                      "periodo", "idade_consulta", "faixa_etaria"] for c in discretizada},
}).sort_values("V")

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(assoc.index, assoc.V, color=[ORANGE if n else BLUE for n in assoc.nova], height=0.65)
for i, v in enumerate(assoc.V):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=7, color=INK)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=ORANGE), plt.Rectangle((0, 0), 1, 1, color=BLUE)],
          labels=["criada/corrigida neste notebook", "original da base"], frameon=False, fontsize=8, loc="lower right")
ax.set_title("Associação com o no-show (V de Cramér, variáveis discretizadas)")
ax.set_xlim(0, assoc.V.max() * 1.2); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Redundância entre variáveis:** pares com V alto carregam quase a mesma informação. Uma Rede Bayesiana com os dois
como pais do no-show "conta duas vezes" a evidência e infla as CPTs. O Manto de Markov tende a reter só um de cada
par, e esta matriz antecipa quais são.

Há dois tipos de redundância: **semântica** (mês × estação, idade × faixa etária × acompanhante, chuva numérica ×
categoria) e **estrutural**, criada pelos blocos de ausência. Quando o cadastro falta, idade, cidade, deficiência e
histórico ficam todos "ausentes" ao mesmo tempo, por isso `cadastro_ausente` aparece ligado a quase tudo.

In [ ]:
nomes = list(discretizada.columns)
mat = pd.DataFrame(np.eye(len(nomes)), index=nomes, columns=nomes)
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        mat.loc[a, b] = mat.loc[b, a] = cramers_v(discretizada[a], discretizada[b])

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(mat, cmap=SEQ, vmin=0, vmax=1, square=True, ax=ax, cbar_kws={"shrink": 0.6, "label": "V de Cramér"},
            linewidths=0.5, linecolor="white")
ax.set_title("Redundância entre variáveis candidatas"); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

pares = mat.where(np.triu(np.ones(mat.shape, dtype=bool), 1)).stack().sort_values(ascending=False)
print("15 pares mais redundantes:")
print(pares.head(15).round(3).to_string())

## 8. Base de variáveis

A base com todas as variáveis candidatas (ainda **contínuas**, sem discretização, que acontece por fold) é salva em
`data/processed/features.parquet` para os notebooks 03 e 04. Ela inclui o alvo, `patient_id` (para a validação
agrupada) e `appointment_date` (para a validação temporal).

In [ ]:
from pathlib import Path

saida = Path.cwd().parent / "data" / "processed" if Path.cwd().name == "notebooks" else Path("data/processed")
saida.mkdir(parents=True, exist_ok=True)
final = df[["appointment_date", "patient_id", TARGET, "city"] + list(FEATURES)]
assert final.equals(build_features()[final.columns]), "notebook e módulo devem gerar a mesma base"
final.to_parquet(saida / "features.parquet", index=False)
print(f"Salvo: {saida / 'features.parquet'} — {final.shape[0]:,} linhas × {final.shape[1]} colunas")

## 9. Síntese

In [ ]:
top = assoc.sort_values("V", ascending=False).head(5)
print("Variáveis mais associadas ao no-show:", ", ".join(f"{i} ({v:.3f})" for i, v in top.V.items()))
r = controle["razão sim/não"]
print(f"Quem faltou na consulta anterior falta {r.min():.1f}× a {r.max():.1f}× mais, em todos os períodos")

| # | Decisão | Motivo |
|---|---------|--------|
| 1 | `idade_consulta` substitui `age`, `under_12` e `over_60` | `age` é a idade na extração da base |
| 2 | Histórico do paciente só com dias anteriores | Desfechos do mesmo dia são homogêneos em 93% dos casos |
| 3 | "ausente" como categoria; `cadastro_ausente` explícito | Ausência de cadastro está associada ao no-show; isso não identifica MAR/MNAR |
| 4 | 17h e 18h em faixas separadas | Extremos opostos de taxa de no-show |
| 5 | `periodo` no lugar de `ano` | Representar a deriva sem um valor que o modelo nunca veria no futuro |
| 6 | Pares redundantes mantidos como candidatos | A seleção fica a cargo do Manto de Markov (notebook 03) |
| 7 | Discretização por quantis ajustada por fold | Evitar vazamento entre treino e teste |

**Próximo notebook:** `03_rede_bayesiana.ipynb`, que vai aprender a estrutura da rede, identificar o Manto de
Markov do no-show e comparar a rede completa com a rede reduzida (H2).